<!-- COMMONS LAUNCHER v3 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building Synapsa, an AI-native
learning platform.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T00-L01-the-8gb-track/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T00-L01-the-8gb-track/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=lessons/T00-L01-the-8gb-track/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer with numpy and matplotlib, which Colab, Kaggle,
Binder and Codespaces already have.

In [ ]:
# --- COMMONS LAUNCHER v3 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = []            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = []    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/lessons/T00-L01-the-8gb-track/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# T00-L01 · The 8 GB track: measure before you believe

**You will build:** `measure()`, a profiler that reports what a piece of code actually
costs, and `tier_check()`, the gate this repository runs against every lesson it ships.

**Time:** ~40 minutes · **Runs on:** a laptop CPU, 8 GiB RAM, no GPU, no download
· **Prerequisites:** none — this is the first lesson in Synapsa Commons.

Every other lesson here claims it fits in 8 GiB and ten minutes. This is the lesson that
makes those claims checkable, so it comes first.

By the end you will be able to:

1. Implement `measure(fn)` reporting wall time, Python-allocator peak and process RSS.
2. Measure the gap between what `tracemalloc` sees and what the operating system charges.
3. Profile four implementations of one task and rank them on time, memory *and* correctness.
4. Implement `tier_check()` so it reproduces this repository's own gate, boundary included.
5. Explain why a lesson that busts its budget gets rewritten, not given a bigger budget.

In [ ]:
# Setup: everything the lesson needs, in one cell, with versions printed.
import contextlib
import io
import mmap
import resource
import subprocess
import sys
import time
import traceback
import tracemalloc
from typing import Any, Callable, Mapping, NamedTuple

import numpy as np

MIB = 1024 * 1024
_LESSON_T0 = time.perf_counter()
print("python", sys.version.split()[0], "· numpy", np.__version__, "· platform", sys.platform)

# The two clocks this lesson is careful to keep apart, described by the interpreter itself
# rather than by me. One of them cannot go backwards; the other one can.
_PERF, _WALL = time.get_clock_info("perf_counter"), time.get_clock_info("time")
print(f"perf_counter: monotonic={_PERF.monotonic}, resolution {_PERF.resolution:.0e}s"
      f"  ·  time(): monotonic={_WALL.monotonic}")

# `ru_maxrss` is not in the same unit everywhere: the Linux man page documents kibibytes and
# the macOS and FreeBSD man pages say kilobytes, while a Darwin kernel is widely reported to
# hand back bytes. Guessing is a good way to be wrong by a factor of 1024, so the next section
# takes nobody's word for it — mine included. It calibrates this divisor against an allocation
# of known size, and says out loud whether the calibration held.
RU_MAXRSS_DIVISOR = MIB if sys.platform == "darwin" else 1024


def rss_hwm_mib() -> float:
    """Peak resident set size of THIS process so far, in MiB.

    This is a high-water mark: it only ever goes up. Freeing memory does not lower it.
    """
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / RU_MAXRSS_DIVISOR


def touch_mmap(mib: int) -> int:
    """Map `mib` MiB of anonymous memory and write to every page, so it is really resident.

    The pages come from the operating system, never from Python's allocator, which is the
    whole point: `tracemalloc` cannot see them and the OS charges you for all of them.
    """
    page = b"\xff" * MIB
    buffer = mmap.mmap(-1, mib * MIB)
    try:
        for _ in range(mib):
            buffer.write(page)
        return len(buffer)
    finally:
        buffer.close()


_FAILED_CHECKS: list[str] = []
_STATUS: dict[str, str] = {}  # label -> "passed" | "failed" | "not started"

# The exercises, in the order you meet them. The progress board at the foot of the
# notebook is built from this, and a demo that is waiting on one names it from here.
_EXERCISES = {
    "exercise 1": "measure()",
    "exercise 2": "chunk_bounds() and sum_squares_chunked()",
    "exercise 3": "profile()",
    "exercise 4": "tier_check()",
}


def _try(label: str, check: Callable[[], None], needs: tuple[str, ...] = ()) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet simply says so. A wrong answer prints the check's own
    message — which names the likely mistake — and the notebook carries on to the next cell,
    so one broken exercise never hides the feedback on the other three.

    A demo names the exercises it `needs`. Until each of them has passed its check, the demo
    says which one it is waiting for and skips, rather than failing half-way through its
    output. Every outcome is recorded in `_STATUS`, which the progress board at the foot reads.
    """
    waiting = [name for name in needs if _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        todo = ", ".join(f"{name} ({_EXERCISES[name]})" for name in waiting)
        print(f"{label}: skipped — this needs {todo} to pass first.")
        return
    try:
        check()
    except NotImplementedError as exc:
        _STATUS[label] = "not started"
        stub = traceback.extract_tb(exc.__traceback__)[-1].name
        print(f"{label}: not implemented yet — fill in {stub}() above, then re-run this cell.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED — {exc}")
    except Exception as exc:  # a half-finished implementation raising something else
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"


_MARKS = {"passed": "✅", "failed": "❌", "not started": "⏳"}


def _progress_board() -> None:
    """One line per exercise, marked passed, failed or not started, then the tally."""
    width = max(len(what) for what in _EXERCISES.values())
    print("progress board")
    for label, what in _EXERCISES.items():
        state = _STATUS.get(label, "not started")
        print(f"  {_MARKS[state]} {label:<11} {what:<{width}}  {state}")
    done = sum(_STATUS.get(label) == "passed" for label in _EXERCISES)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    failing = [label for label, state in _STATUS.items() if state == "failed"]
    if failing:
        print("failing right now: " + ", ".join(failing) + ". Each one printed what went wrong "
              "in its own cell above, and every exercise heading has hints you can open.")
    elif done < len(_EXERCISES):
        print("work top to bottom: every exercise heading has hints you can open.")


print(f"this process has already peaked at {rss_hwm_mib():.1f} MiB just by starting up")

## 1. The phenomenon: two honest tools that disagree

Python ships two ways to ask "how much memory did that cost?", and they answer different
questions.

- `tracemalloc` traces memory blocks **allocated by Python**. Precise, per-call, and blind
  to anything that never passes through Python's allocators.
- `resource.getrusage(...).ru_maxrss` is the **operating system's** high-water mark for the
  whole process. It sees everything, forgets nothing, and cannot be scoped to one call.

Run this. The workload allocates a known amount of memory outside Python's allocator.

In [ ]:
tracemalloc.start()
tracemalloc.reset_peak()
_rss_before = rss_hwm_mib()
_mapped = touch_mmap(256)
_py_peak = tracemalloc.get_traced_memory()[1] / MIB
_rss_after = rss_hwm_mib()
tracemalloc.stop()

print(f"asked the OS for        {_mapped / MIB:.1f} MiB, and wrote to every page")
print(f"tracemalloc saw         {_py_peak:.2f} MiB")
print(f"the OS high-water mark  rose by {_rss_after - _rss_before:.1f} MiB")

_calibration_error = abs((_rss_after - _rss_before) - _mapped / MIB)
print(f"\ncalibration: divisor {RU_MAXRSS_DIVISOR} reproduces a known {_mapped / MIB:.0f} MiB "
      f"allocation to within {_calibration_error:.1f} MiB")
print("  → the unit is right on this machine" if _calibration_error < 32 else
      "  → the unit is WRONG on this machine: every MiB figure below is off by a constant\n"
      "    factor, so fix RU_MAXRSS_DIVISOR before believing anything this notebook prints")

Neither tool is lying. `tracemalloc` reported honestly on the blocks Python allocated —
there were almost none. The number that decides whether a lesson ships is the second one,
because it is the one that makes a laptop swap and a free notebook tier kill the kernel.

A profiler worth having reports both, plus the clock. That is exercise 1.

## 2. Exercise 1 — `measure()`

Fill in the function. `Measurement` is given to you; you supply how each field is obtained.

Three traps are deliberately in your way:

- `time.time()` can go **backwards** when the system clock is adjusted. Use
  `time.perf_counter()`, which is monotonic and includes time spent asleep.
- `tracemalloc`'s peak is cumulative while tracing runs, so a second call inherits the
  first call's peak until something resets it.
- `ru_maxrss` never comes down, so the *rise* during your call is only meaningful when your
  call set a new record for the whole process.

<details><summary>💡 Hint 1 — what to think about</summary>

Five fields, five sources. For each one ask whether an *earlier* call can leak into it:
a traced peak that nothing reset, a high-water mark that never falls. Then ask what the
clock and the tracer should do when `fn` raises, and whether a `Measurement` should
exist at all in that case.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Read the process high-water mark, switch tracing on and reset its peak, then start a
`perf_counter` clock. Call `fn` inside a `try` whose `finally` stops the clock, keeps the
*peak* half of the traced-memory pair and switches tracing off — no `except`, so a
failure comes straight back out. Then read the high-water mark again: that reading is
one field, and its rise over the first one, never below zero, is another. Bytes become
MiB through `MIB`.
</details>

In [ ]:
class Measurement(NamedTuple):
    """What one run of one callable cost."""

    result: Any           # whatever fn returned, so measuring is never in the way
    wall_s: float         # elapsed seconds on a monotonic clock
    py_peak_mib: float    # peak memory Python's own allocator held during the call
    rss_hwm_mib: float    # the whole process's high-water RSS after the call, absolute
    rss_delta_mib: float  # how much that high-water mark rose during the call, never negative


def measure(fn: Callable[..., Any], *args: Any, **kwargs: Any) -> Measurement:
    """Call `fn(*args, **kwargs)` and report what it cost.

    `result` must be exactly what `fn` returned, so a measured call can be dropped in
    anywhere the plain call was.

    Requirements, each of which is graded:
      * `wall_s` comes from `time.perf_counter()`, not `time.time()`.
      * `py_peak_mib` is this call's own peak in MiB. Whichever route you take — resetting the
        peak once tracing is on, stopping tracing when the call returns, or both — a later
        small call must never report a big earlier call's number.
      * `rss_hwm_mib` is the process high-water mark after the call, in MiB.
      * `rss_delta_mib` is `rss_hwm_mib` minus the same reading taken before the call,
        clamped at 0.0.
      * if `fn` raises, the exception comes straight back out: a measured call is a drop-in
        replacement for the plain one. A `finally` is the tidy way to stop the clock and
        switch tracing off on both paths; an `except` that returns a Measurement anyway
        reports a cost for work that never finished.

    Example:
        >>> m = measure(sum, [1, 2, 3])
        >>> m.result
        6
        >>> m.wall_s < 1.0 and m.rss_hwm_mib > 0.0
        True
    """
    # YOUR CODE HERE
    raise NotImplementedError


# Public checks — run these as often as you like.
def _check_measure() -> None:
    passthrough = measure(sum, [1, 2, 3])
    assert isinstance(passthrough, Measurement), (
        "measure must return a Measurement, not a tuple, dict or bare number — "
        "build it with Measurement(result=..., wall_s=..., ...)"
    )
    assert passthrough.result == 6, (
        "measure lost the return value: call fn(*args, **kwargs), keep what it returns, "
        "and put it in Measurement.result"
    )

    slow = measure(time.sleep, 0.25)
    assert 0.2 <= slow.wall_s <= 1.5, (
        f"measure(time.sleep, 0.25) reported {slow.wall_s:.3f}s — near zero means you are "
        "timing the wrong thing; ~250 means you are reporting milliseconds, not seconds"
    )

    big = measure(bytearray, 48 * MIB)
    assert 40.0 <= big.py_peak_mib <= 80.0, (
        f"a 48 MiB bytearray showed as {big.py_peak_mib:.1f} MiB — are you dividing bytes by "
        "1024*1024, and reading the PEAK (second element) of get_traced_memory()?"
    )

    small = measure(bytearray, 1024)
    assert small.py_peak_mib < 5.0, (
        f"a 1 KiB allocation reported {small.py_peak_mib:.1f} MiB, which is the previous "
        "call's peak — reset it with tracemalloc.reset_peak() once tracing is on, and call "
        "tracemalloc.stop() when the call returns"
    )

    assert big.rss_hwm_mib > 5.0, (
        "rss_hwm_mib is the whole process's peak, so it is tens of MiB even for a tiny call; "
        "a near-zero value means you subtracted a baseline that belongs in rss_delta_mib"
    )
    assert small.rss_delta_mib >= 0.0, "rss_delta_mib must be clamped at 0.0, never negative"

    class _Boom(RuntimeError):
        pass

    def _explode() -> None:
        raise _Boom("this call was meant to fail")

    try:
        measure(_explode)
    except _Boom:
        pass
    else:
        raise AssertionError(
            "measure() swallowed the exception _explode raised and returned a Measurement "
            "anyway — that reports a cost for work that never finished. Catch nothing; put "
            "the clock and the tracemalloc clean-up in a finally block instead"
        )
    print("exercise 1 looks right — measure() reports the clock, Python's peak and the OS peak")

In [ ]:
_try("exercise 1", _check_measure)

### What `measure()` can and cannot tell you

Run the two cells below once `measure()` works. The first is the high-water trap: the *same*
workload, measured twice, and neither answer is the amount of memory it used. The second
re-proves the blind spot through your own profiler. A `rss_delta_mib` of 0.0 never means
"this was free"; it means "this set no new record".

In [ ]:
def _show_high_water_trap() -> None:
    asked = 512
    first = measure(touch_mmap, asked)
    second = measure(touch_mmap, asked)
    print(f"first  touch_mmap({asked}): rose {first.rss_delta_mib:6.1f} MiB "
          f"(process peak now {first.rss_hwm_mib:.0f} MiB)")
    print(f"second touch_mmap({asked}): rose {second.rss_delta_mib:6.1f} MiB "
          f"(process peak now {second.rss_hwm_mib:.0f} MiB)")
    print(f"\nsame work, two answers, and neither of them is {asked} MiB:")
    print("  the first understates it, because the mark did not start from the floor;")
    print("  the second reports nothing at all, because it set no new record.")
    print("a high-water mark measures records, not usage — which is why the repository's")
    print("gate runs every lesson in a FRESH process, where the record starts at the floor.")


_try("high-water demo", _show_high_water_trap, needs=("exercise 1",))

In [ ]:
def _show_blind_spot() -> None:
    m = measure(touch_mmap, 256)
    print(f"asked the OS for      {m.result / MIB:.0f} MiB, and wrote to every page")
    print(f"tracemalloc saw       {m.py_peak_mib:.2f} MiB of it")
    print(f"process high-water is {m.rss_hwm_mib:.0f} MiB")
    print(f"the call took         {m.wall_s * 1000:.0f} ms")
    print("\nwhat tracemalloc did see is the reusable page buffer inside touch_mmap, which is")
    print("a genuine Python bytes object. The mapping itself never reached Python's allocator,")
    print("so tracemalloc cannot report it — and the gate charges you for it regardless.")


_try("blind-spot demo", _show_blind_spot, needs=("exercise 1",))

## 3. The floor you never asked for

A fresh interpreter costs memory before your code runs at all, and importing a library costs
more. On an 8 GiB tier that floor is not yours to spend. Measure it where the repository's
own gate measures it — in a fresh child interpreter, never in this one. `tools/execute.py`
reads the child's own `RUSAGE_SELF` from inside it; from out here the same peak arrives
through `RUSAGE_CHILDREN`.

The snippets run in ascending order of cost on purpose. `RUSAGE_CHILDREN` is also a
high-water mark, over *all* finished children, so a cheap child measured after an expensive
one reports the expensive one's number. Same trap, one level up.

In [ ]:
def measure_subprocess(snippet: str) -> tuple[float, float]:
    """Run `snippet` in a fresh interpreter; return (wall seconds, peak child RSS in MiB).

    The figure returned is the absolute high-water mark over all children, not the rise
    during this one. A difference would be the honest number if the mark could fall, and it
    cannot: that is the trap this whole section is about, one level up. Running the snippets
    in ascending order of cost is what keeps each reading its own.
    """
    t0 = time.perf_counter()
    proc = subprocess.run([sys.executable, "-c", snippet], capture_output=True, text=True)
    wall = time.perf_counter() - t0
    after = resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss
    if proc.returncode != 0:
        raise RuntimeError(f"child failed: {proc.stderr.strip().splitlines()[-1:]}")
    return wall, after / RU_MAXRSS_DIVISOR


for _label, _snippet in (("a bare interpreter", "pass"),
                         ("+ import numpy", "import numpy"),
                         ("+ import numpy, json, urllib.request",
                          "import numpy, json, urllib.request")):
    _wall, _peak = measure_subprocess(_snippet)
    print(f"{_label:38s} {_peak:7.1f} MiB   {_wall * 1000:5.0f} ms to start")

## 4. Exercises 2 and 3 — profile one task, four ways

The task: the exact sum of `i * i` for `i` in `range(n)`. Three implementations are given.
They differ in memory and in speed — and one of them is **silently wrong** at the size we
are about to run, which no profiler would ever have told you.

You will write the fourth.

In [ ]:
N_DEMO = 4_000_000


def exact_sum_squares(n: int) -> int:
    """Closed form, for checking the others: the sum of i*i for i in range(n)."""
    return (n - 1) * n * (2 * n - 1) // 6


def sum_squares_list(n: int) -> int:
    """Materialise every square in a list, then add them up."""
    return sum([i * i for i in range(n)])


def sum_squares_generator(n: int) -> int:
    """Same arithmetic, one square alive at a time."""
    return sum(i * i for i in range(n))


def sum_squares_numpy_whole(n: int) -> int:
    """One big int64 array, squared and summed by numpy."""
    values = np.arange(n, dtype=np.int64)
    return int((values * values).sum())


_truth = exact_sum_squares(N_DEMO)
print(f"int64 holds up to   {np.iinfo(np.int64).max}")
print(f"the exact answer is {_truth}\n")
print(f"{'implementation':22s} {'time':>10s} {'returns':>21s} {'exact?':>7s}")
for _name, _impl in (("list of squares", sum_squares_list),
                     ("generator", sum_squares_generator),
                     ("numpy, whole array", sum_squares_numpy_whole)):
    _t0 = time.perf_counter()
    _got = _impl(N_DEMO)
    _dt = time.perf_counter() - _t0
    print(f"{_name:22s} {_dt:8.3f} s {_got:>21} {str(_got == _truth):>7s}")

Read the table you just produced: the quickest of the three is the only one that is wrong,
and nothing about the way it failed looks like failure. The running total overflowed int64
and wrapped around, where Python's own `int` has no ceiling to wrap at. No profiler would
have caught it, because a profiler answers "what did that cost", never "was that right".

So the fourth implementation has to get numpy's speed and bounded memory *and* Python's
exactness: work in slices, and accumulate the running total in a Python `int`.

Two stubs: the slice boundaries first, then the sum that uses them.

<details><summary>💡 Exercise 2 · Hint 1 — what to think about</summary>

Walk three ranges on paper before writing a loop: one the chunk divides exactly, one
it does not, and an empty one. What would a chunk of zero do to your loop? For the sum,
ask what type your running total is after you add a numpy scalar to it.
</details>
<details><summary>💡 Exercise 2 · Hint 2 — the approach, in words</summary>

Step a start index from the front of the range towards `n` in strides of `chunk`; each
stop is the start plus a chunk, capped at `n`. Refuse a non-positive chunk before the
loop begins. For the sum, build only the current slice as int64, square and total it,
turn that one slice total into a Python `int`, and add it to a running total that is a
plain Python `int` from the start.
</details>

<details><summary>💡 Exercise 3 · Hint 1 — what to think about</summary>

All the measuring is already done by `measure()`. What `profile` must protect is the
shape of the answer: the keys it was handed, every implementation run on the same `n`,
and nothing carried over from a previous call.
</details>
<details><summary>💡 Exercise 3 · Hint 2 — the approach, in words</summary>

Build a fresh dict inside the function — never a default argument or a module-level
one — and for each name and implementation store what `measure` returns when you hand
it that implementation and `n`. Return it without sorting, printing or dropping.
</details>

In [ ]:
def chunk_bounds(n: int, chunk: int) -> list[tuple[int, int]]:
    """Split range(n) into consecutive half-open [start, stop) slices of at most `chunk`.

    The slices must cover range(n) exactly once, in order, with no gaps and no overlap.
    Raise ValueError if `chunk` is not positive.

    Example:
        >>> chunk_bounds(10, 4)
        [(0, 4), (4, 8), (8, 10)]
        >>> chunk_bounds(0, 4)
        []
    """
    # YOUR CODE HERE
    raise NotImplementedError


def sum_squares_chunked(n: int, chunk: int = 500_000) -> int:
    """Exact sum of i*i for i in range(n), never holding more than `chunk` elements.

    Build each slice with `np.arange(start, stop, dtype=np.int64)`, square it, sum it, and add
    that slice's total into a running Python `int`. Accumulating in a Python int is what keeps
    the answer exact once the total passes int64's ceiling.

    Example:
        >>> sum_squares_chunked(7, chunk=3)
        91
        >>> sum_squares_chunked(0)
        0
    """
    # YOUR CODE HERE
    raise NotImplementedError


def profile(impls: Mapping[str, Callable[[int], int]], n: int) -> dict[str, Measurement]:
    """Measure every implementation in `impls` on the same input `n`.

    Return a dict with the same keys, each mapped to the `Measurement` from calling that
    implementation with `n`. Do not sort, do not print, do not drop the results.

    Example:
        >>> table = profile({"exact": exact_sum_squares}, 7)
        >>> table["exact"].result
        91
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_chunking() -> None:
    assert chunk_bounds(10, 4) == [(0, 4), (4, 8), (8, 10)], (
        "chunk_bounds(10, 4) must be [(0, 4), (4, 8), (8, 10)] — the last slice is short, "
        "and stop is exclusive"
    )
    assert chunk_bounds(0, 4) == [], "chunk_bounds(0, 4) is an empty list, not [(0, 0)]"
    assert chunk_bounds(8, 4) == [(0, 4), (4, 8)], (
        "when chunk divides n exactly there is no extra empty slice at the end"
    )
    try:
        chunk_bounds(10, 0)
    except ValueError:
        pass
    else:
        raise AssertionError("chunk_bounds(10, 0) must raise ValueError, not loop forever")

    assert sum_squares_chunked(7, chunk=3) == 91, (
        "sum_squares_chunked(7, chunk=3) should be 91 — do your slices cover range(n) "
        "exactly once?"
    )
    assert sum_squares_chunked(0) == 0, "an empty range sums to 0, not an error"
    big = sum_squares_chunked(N_DEMO)
    assert big == exact_sum_squares(N_DEMO), (
        f"off by {big - exact_sum_squares(N_DEMO)} at n={N_DEMO}: each slice's sum is fine, "
        "but the running total must be a Python int — add int(slice_total) to an int, do not "
        "accumulate into a numpy int64"
    )
    print("exercise 2 looks right — the chunked sum is exact at a size where numpy overflows")


def _check_profile() -> None:
    table = profile({"exact": exact_sum_squares, "generator": sum_squares_generator}, 7)
    assert set(table) == {"exact", "generator"}, (
        "profile must return one entry per implementation, keyed by the same names"
    )
    assert all(isinstance(m, Measurement) for m in table.values()), (
        "each value must be the Measurement returned by measure(), not just a duration"
    )
    assert table["exact"].result == 91, (
        "profile must call each implementation with n and keep its result — "
        "measure(fn, n) already does that for you"
    )
    print("exercise 3 looks right — profile() returns a Measurement per implementation")

In [ ]:
_try("exercise 2", _check_chunking)
_try("exercise 3", _check_profile, needs=("exercise 1",))

Now the trade-off, measured rather than asserted. Every number below is produced by your
`profile()`; none of it was typed into this notebook.

In [ ]:
def _show_tradeoff() -> None:
    impls = {
        "list of squares": sum_squares_list,
        "generator": sum_squares_generator,
        "numpy, whole array": sum_squares_numpy_whole,
        "numpy, chunked": sum_squares_chunked,
    }
    table = profile(impls, N_DEMO)
    truth = exact_sum_squares(N_DEMO)
    print(f"{'implementation':22s} {'time':>10s} {'py peak':>12s} {'exact?':>7s}")
    for name, m in table.items():
        print(f"{name:22s} {m.wall_s:8.3f} s {m.py_peak_mib:8.1f} MiB {str(m.result == truth):>7s}")
    fastest = min((m.wall_s, name) for name, m in table.items() if m.result == truth)
    leanest = min((m.py_peak_mib, name) for name, m in table.items() if m.result == truth)
    print(f"\nfastest correct: {fastest[1]} at {fastest[0]:.3f}s")
    print(f"leanest correct: {leanest[1]} at {leanest[0]:.1f} MiB")
    whole, chunked = table["numpy, whole array"], table["numpy, chunked"]
    print(f"whole array vs chunked: {whole.py_peak_mib / max(chunked.py_peak_mib, 1e-9):.1f}x "
          f"the memory, {whole.wall_s / max(chunked.wall_s, 1e-9):.1f}x the time, and "
          f"exact={whole.result == truth} against exact={chunked.result == truth}")


_try("trade-off table", _show_tradeoff, needs=("exercise 1", "exercise 2", "exercise 3"))

### The profiler is not free

One row of that table is badly distorted, and `measure()` is what distorted it. Tracing
every allocation costs time, so the implementation that allocates four million objects pays
a tax the one that allocates almost none never sees. Measure the same function twice — once
under tracing, once without it — and see how large the tax is on this machine.

In [ ]:
def _show_observer_effect() -> None:
    t0 = time.perf_counter()
    sum_squares_list(N_DEMO)
    untraced = max(time.perf_counter() - t0, 1e-9)
    traced = measure(sum_squares_list, N_DEMO).wall_s
    print(f"sum_squares_list, tracemalloc off: {untraced:.3f}s")
    print(f"sum_squares_list, tracemalloc on:  {traced:.3f}s")
    print(f"tracing made this workload {traced / untraced:.1f}x slower")
    print("\nthe ranking in the table above survives this; the absolute seconds do not.")
    print("tools/execute.py times a lesson with tracing OFF, in a fresh process, which is why")
    print("its wall-clock number — not this one — is what the budget is written against.")


_try("observer effect", _show_observer_effect, needs=("exercise 1",))

## 5. Exercise 4 — `tier_check()`, the gate itself

This repository sorts every lesson into a compute tier and refuses to ship one that does not
fit. `cpu8` — the tier this lesson runs in — means 8 GiB and ten minutes. `phone` and
`browser` mean 2 GiB. The GPU tiers mean what their names say.

Reproduce the gate. The rules, in order:

1. An unknown tier fails immediately with exactly one reason: there is no ceiling to compare
   against, so nothing else is checked.
2. A budget that is not positive fails. An undeclared budget is not an infinite one.
3. Wall time above the declared budget fails.
4. Peak memory above the tier's ceiling fails.
5. Being exactly *at* a limit passes. The comparison is `>`, not `>=`.

<details><summary>💡 Hint 1 — what to think about</summary>

Only one rule may stop the checking early: the one with no ceiling to compare against.
Every other broken rule adds its own reason, in order — so what happens to a run that
is over on time *and* on memory if you return at the first problem you find? And is a
run that lands exactly on a limit over it?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Look the tier up first; if it is not there, return a failing verdict with that single
reason. Otherwise start an empty list and test the budget, the time and the memory in
rule order, appending a reason for each broken rule. Format the budget in the time
reason with `:.3g`, so a sub-second one stays legible, but put the tier's ceiling into
the memory reason as the table holds it: `:.3g` would print a four-digit ceiling in
scientific notation. It passes exactly when the list stayed empty; hand the reasons
back as a tuple.
</details>

In [ ]:
TIER_MIB = {
    "phone": 2048,
    "browser": 2048,
    "cpu8": 8192,
    "free-gpu": 16384,
    "gpu24": 24576,
    "gpu80": 81920,
    "multi-gpu": 163840,
    "api": 8192,
}


class Verdict(NamedTuple):
    """The gate's answer: did it pass, and if not, every reason why not."""

    passed: bool
    reasons: tuple[str, ...]


def tier_check(measured: Mapping[str, float], tier: str, budget_seconds: float) -> Verdict:
    """Decide whether a measured run fits its declared tier and budget.

    `measured` is a mapping with the keys "wall_s" and "peak_mib". Return a `Verdict` whose
    `reasons` is an empty tuple when it passes, and otherwise holds one human-readable string
    per broken rule, in the rule order given above. A memory reason must contain the tier's
    ceiling and a time reason the budget, so the author can see how far over they are. Format
    those numbers so they survive a small budget: `f"{budget_seconds:.1f}"` turns 0.05 into
    "0.1", which is how a gate ends up reporting "0.2s exceeds 0.1s". `:.3g` keeps both
    legible at either scale; bare `:g` spends six significant figures getting there.

    Example:
        >>> tier_check({"wall_s": 12.0, "peak_mib": 900.0}, "cpu8", 600).passed
        True
        >>> tier_check({"wall_s": 12.0, "peak_mib": 9000.0}, "cpu8", 600).passed
        False
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_tier_check() -> None:
    ok = tier_check({"wall_s": 12.0, "peak_mib": 900.0}, "cpu8", 600)
    assert ok.passed and ok.reasons == (), (
        "a run well inside its tier must pass with an EMPTY reasons tuple"
    )
    edge = tier_check({"wall_s": 600.0, "peak_mib": 8192.0}, "cpu8", 600)
    assert edge.passed, (
        "exactly at the limit passes: compare with > (over), not >= (at or over)"
    )
    unknown = tier_check({"wall_s": 1.0, "peak_mib": 1.0}, "cpu9000", 600)
    assert not unknown.passed and len(unknown.reasons) == 1, (
        "an unknown tier fails with exactly one reason and stops there — there is no ceiling "
        "to compare memory against"
    )
    both = tier_check({"wall_s": 900.0, "peak_mib": 9000.0}, "cpu8", 600)
    assert not both.passed and len(both.reasons) == 2, (
        f"over on both axes should give 2 reasons, got {len(both.reasons)}: append one reason "
        "per broken rule instead of returning at the first one"
    )
    assert "8192" in both.reasons[1], (
        "the memory reason must name the ceiling it broke, so the author can see the gap"
    )
    print("exercise 4 looks right — tier_check reproduces the gate, boundary included")

In [ ]:
_try("exercise 4", _check_tier_check)

### Run the gate on this very notebook

Nothing below is hypothetical. It measures this process and asks your own gate whether this
lesson may ship.

In [ ]:
def _gate_this_lesson() -> None:
    elapsed = time.perf_counter() - _LESSON_T0
    peak = rss_hwm_mib()
    verdict = tier_check({"wall_s": elapsed, "peak_mib": peak}, "cpu8", 90)
    print(f"this notebook so far: {elapsed:.1f}s and {peak:.0f} MiB peak")
    print(f"tier_check says: passed={verdict.passed} reasons={verdict.reasons}")


_try("gate on this lesson", _gate_this_lesson, needs=("exercise 4",))

## 6. Why the budget does not move

When a lesson comes in over its tier there are two things an author can do, and only one of
them is allowed here.

Raising the budget keeps the author comfortable and moves the cost onto the student. It is
invisible in review — the gate goes green — and it is paid by exactly the people who cannot
argue back: the ones on a four-year-old laptop, a shared machine, a free notebook tier that
kills the kernel at its own ceiling. A lesson that needs 12 GiB does not teach 8 GiB
students slowly. It does not teach them at all.

Rewriting is the other option, and it nearly always produces a better lesson: fewer steps, a
smaller model, a shorter horizon, a slice instead of the whole array. The constraint is the
pedagogy, not an obstacle to it. You have already written the proof — `sum_squares_chunked`
holds a fraction of the memory the whole-array version needs, gives up nothing in speed to
do it (your own table printed the ratio), and is exact at a size where the version it
replaced silently wraps. Shrinking the problem cost nothing. It found a bug.

The budget is a promise to a student you will never meet. Measure, then keep it.

Run it rather than take my word for it. Below, one task is judged twice against the `phone`
tier — as first written, then rewritten — and the rewrite passes a gate the original fails
while returning an identical answer.

In [ ]:
def _show_why_the_budget_does_not_move() -> None:
    n = N_DEMO
    budget = 0.05
    print(f"n={n}, judged against the phone tier ({TIER_MIB['phone']} MiB) "
          f"and a {budget}s budget.")
    print("timed with tracing OFF, the way tools/execute.py times a lesson; the memory figure")
    print("comes from a separate traced run, because there is no per-call RSS number to take —")
    print("a high-water mark belongs to the process, not to the call.\n")
    answers = []
    for name, fn in (("as written (list)", sum_squares_list),
                     ("rewritten (chunked)", sum_squares_chunked)):
        t0 = time.perf_counter()
        answers.append(fn(n))
        untraced = time.perf_counter() - t0
        py_peak = measure(fn, n).py_peak_mib
        verdict = tier_check({"wall_s": untraced, "peak_mib": py_peak}, "phone", budget)
        print(f"  {name:22s} {untraced:6.3f}s {py_peak:7.1f} MiB  passed={verdict.passed}")
        for reason in verdict.reasons:
            print(f"  {'':22s}   {reason}")
    print(f"\nboth return the same answer: {answers[0] == answers[1]}")
    print("the rewrite passes a gate the original fails, and costs the student nothing.")
    print("a bigger budget would have hidden that difference instead of finding it.")


_try("budget versus rewrite", _show_why_the_budget_does_not_move,
     needs=("exercise 1", "exercise 2", "exercise 4"))

## 7. Common mistakes

- **Timing with `time.time()`.** It follows the system clock and can jump backwards
  mid-run, producing negative durations. `time.perf_counter()` is monotonic.
- **Trusting a zero `rss_delta_mib`.** A high-water mark only moves on a new record. The
  second run of an identical workload usually reports a rise of 0.0.
- **Believing `tracemalloc` is the whole story.** It reports what Python allocated. Memory
  mapped straight from the OS is invisible to it, and the gate counts it anyway.
- **Reporting `get_traced_memory()[0]`.** That is the *current* size; the peak is `[1]`.
- **Catching what the measured call raised.** A profiler that turns a failure into a default
  `Measurement` reports a cost for work that never happened, and the run goes green.
  `try`/`finally` stops the clock on both paths; `try`/`except` hides one of them.
- **Measuring once.** The first run pays import and page-fault costs the second never sees.
- **Letting numpy pick the accumulator.** `int64` wraps silently. Sum slices into a Python
  `int` whenever the total can outgrow the dtype.
- **Profiling in the same process as everything else.** Measure in a fresh interpreter, the
  way `tools/execute.py` does, or the previous cell's peak becomes your result.
- **Quoting a traced timing in a budget.** `tracemalloc` taxes allocation-heavy code far
  more than it taxes numpy, so timings taken under tracing are not comparable to timings
  taken without it. Time with tracing off; trace memory in a separate run.

The fourth one is worth seeing rather than believing. Run the cell below: it allocates a
32 MiB buffer and throws it away inside a single call, which is what almost every real
function does with its working memory.

In [ ]:
def _show_current_versus_peak() -> None:
    tracemalloc.start()
    tracemalloc.reset_peak()
    blob = bytearray(32 * MIB)
    alive = tracemalloc.get_traced_memory()
    del blob
    freed = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    print(f"while the buffer is alive: current={alive[0] / MIB:6.1f} MiB  peak={alive[1] / MIB:6.1f} MiB")
    print(f"after it is thrown away:   current={freed[0] / MIB:6.1f} MiB  peak={freed[1] / MIB:6.1f} MiB")
    print("\n[0] is `current` and forgets; [1] is `peak` and remembers. A profiler that reads")
    print("[0] after the call has returned will report that this function cost nothing at all.")


_show_current_versus_peak()

## 8. Self-check

1. Your `measure()` reports `rss_delta_mib = 0.0` for a function that builds a 300 MiB
   array. The most likely explanation is:
   - (a) the array was freed before the measurement ended, so it did not count
   - (b) something earlier in this process already peaked higher, so no new record was set
   - (c) `ru_maxrss` does not count numpy arrays

2. `tracemalloc` reports 0.00 MiB for `touch_mmap(256)` while the OS charges 256 MiB. This
   means:
   - (a) `tracemalloc` is broken on this platform
   - (b) the pages were never resident, so nothing was really used
   - (c) the memory never passed through Python's allocator, which is all `tracemalloc` sees

3. `sum_squares_numpy_whole` beat both pure-Python versions by more than an order of
   magnitude and still returned the wrong answer at `n = 4_000_000`. No profiler caught
   that, because:
   - (a) profilers measure cost, not correctness; only a reference answer catches this
   - (b) the error is too small to show up in a timing measurement
   - (c) overflow affects only memory, never results

4. A lesson measures 11 minutes and 9 GiB on the `cpu8` tier. The correct response is:
   - (a) raise `budget_seconds` and declare a larger tier
   - (b) shrink the problem — fewer steps, smaller model, shorter horizon — and re-measure
   - (c) keep the declared tier and note in the README that slower machines may struggle

5. In the trade-off table, `list of squares` looks far slower than it is when you call it
   on its own. The reason is:
   - (a) the list implementation really is that slow; the table is accurate
   - (b) `measure()` traces every allocation, and that implementation makes millions of them
   - (c) numpy released the GIL, so the other implementations were given more CPU

Answers are in this lesson's worked solution in the course repository.

## What you built, and where it goes next

`measure()` and `tier_check()` are the instruments every other lesson in Synapsa Commons is held
to: each one declares a tier, and the executor writes back the numbers it actually observed
instead of the ones its author hoped for. You now own the gate that judges them.

In [ ]:
# Your progress board. Every check is re-run here, quietly, against your code as it stands
# now — each one already printed its feedback in its own cell above — so the board is
# current even if you edited an exercise and did not re-run its check.
if __name__ == "__main__":
    _ALL_CHECKS = (  # (exercise, its check, the exercises that check relies on)
        ("exercise 1", _check_measure, ()),
        ("exercise 2", _check_chunking, ()),
        ("exercise 3", _check_profile, ("exercise 1",)),
        ("exercise 4", _check_tier_check, ()),
    )
    with contextlib.redirect_stdout(io.StringIO()):
        for _name, _check, _needs in _ALL_CHECKS:
            _try(_name, _check, needs=_needs)
    _progress_board()
    # A stub you have not reached yet is not a failure. A check that ran and came back wrong
    # is: in a script or under CI it ends the run non-zero, rather than letting a green exit
    # code paper over it. Inside a notebook kernel the board above has already said so, in a
    # line rather than a traceback at the foot of the page.
    if _FAILED_CHECKS and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(dict.fromkeys(_FAILED_CHECKS)))